In [ ]:
import os
import numpy as np
import pandas as pd
import pickle

from code_libraries.training_metrics import prob_cutoff_confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import sys
!{sys.executable} -m pip install xgboost

from code_libraries.xgb_trainter_lib import XGBoostGS

## Paths

In [ ]:
main_dir  = "../"
save_dir  = os.path.join(main_dir, "heavy_data_NO_GIT/models_scalers")
data_arrays = os.path.join(main_dir, "heavy_data_NO_GIT/data_arrays")


os.makedirs(save_dir, exist_ok=True)
os.makedirs(data_arrays, exist_ok=True)

## Dataset registry All (chain, pt) combos — add/remove rows here as needed

In [ ]:
CHAINS = [ "sn1, sn2"]
PT_SIZES = ["raw", 3, 5, 7, 9, 11, 13, 15, 17]

base_csv_dir = os.path.join(
    main_dir, "heavy_data_NO_GIT/dataframes/sym"
)

## Basic training parameters

In [ ]:
random_state = 76
test_size    = 0.30
results_summary = []


def _pred_from_proba(model, X, cutoff=0.5):
    """Return hard predictions using predict_proba (preferred) or predict()."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            p1 = proba[:, 1]
        else:
            p1 = np.ravel(proba)
        return (p1 >= cutoff).astype(int)
    return model.predict(X)

## Main training loop

In [ ]:
# Keep fitted objects alive for any post-loop use
fitted = {}   # key: (chain, pt)  ->  {"model": ..., "scaler": ..., "X_test": ..., "y_test": ...}

for chain in CHAINS:
    for pt in PT_SIZES:
        suffix = "raw" if pt == "raw" else f"{pt}pt"
        tag = f"{chain}_{suffix}"
        print(f"\n{'='*20} {tag} {'='*20}")

        # ---- Load data ----
        csv_path = os.path.join(base_csv_dir, f"training_{chain}_ft_{suffix}.csv")
        df = pd.read_csv(csv_path)
        if "Unnamed: 0" in df.columns:
            df.drop("Unnamed: 0", axis=1, inplace=True)

        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, shuffle=True, random_state=random_state
        )

        # ---- Scale ----
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        # ---- Train ----
        xgb = XGBoostGS(
            use_scaler=False,       # already scaled above
            halving=True,
            n_splits=5,
            random_state=random_state,
            n_jobs=-1,              # use all CPU cores for CV
            verbose=0,
        )
        xgb.fit(X_train_s, y_train)

        # ---- Evaluate ----
        cm  = prob_cutoff_confusion_matrix(xgb, X_test_s, y_test)
        acc = np.trace(cm) / len(y_test)
        y_pred = _pred_from_proba(xgb, X_test_s, cutoff=0.5)
        f1  = f1_score(y_test, y_pred)

        print(f"XGB {tag} accuracy: {acc:.4f}")
        print(f"XGB {tag} confusion matrix:\n{cm}")
        print(f"XGB {tag} f1: {f1:.4f}")

        results_summary.append({
            "tag": tag,
            "accuracy": float(acc),
            "f1": float(f1),
            "confusion_matrix": cm.tolist(),
        })

        # ---- Save model + scaler ----
        model_path  = os.path.join(save_dir, f"xgb_{tag}_model.sav")
        scaler_path = os.path.join(save_dir, f"xgb_{tag}_scaler.sav")
        pickle.dump(xgb,    open(model_path,  "wb"))
        pickle.dump(scaler, open(scaler_path, "wb"))

        # ---- Save test arrays ----
        test_tuple = (X_test, y_test)
        with open(os.path.join(data_arrays, f"xgb_test_{tag}.pkl"), "wb") as f:
            pickle.dump(test_tuple, f)


        # Keep reference for any post-loop work
        fitted[(chain, pt)] = {
            "model": xgb, "scaler": scaler,
            "X_test": X_test_s, "y_test": y_test,
        }

## Save summary metrics

In [ ]:
results_txt_path = os.path.join(save_dir, "xgb_ft_last_sn1_sn2_metrics.txt")
with open(results_txt_path, "w", encoding="utf-8") as f:
    f.write("XGBoost (FT_LAST) separate sn1/sn2 metrics\n")
    f.write(f"random_state={random_state}, test_size={test_size}\n\n")
    for r in results_summary:
        f.write(f"[{r['tag']}]\n")
        f.write(f"accuracy: {r['accuracy']:.6f}\n")
        f.write(f"f1: {r['f1']:.6f}\n")
        f.write("confusion_matrix:\n")
        cm = np.array(r["confusion_matrix"])
        f.write(str(cm) + "\n\n")

print("\nSaved metrics summary to:", results_txt_path)